# Drone vs Bird, entraînement du modèle de base

**Où tourne ce notebook :** sur Google Colab, dans le navigateur. Rien ne s'installe sur le PC.

**Avant de lancer quoi que ce soit :** menu `Exécution` > `Modifier le type d'exécution` > choisir **T4 GPU** > Enregistrer.

Ensuite, exécuter les cellules une par une, de haut en bas, avec le bouton play à gauche de chacune.


## 1. Vérifier que le GPU est actif

Doit afficher `True` et `Tesla T4`. Si c'est `False`, le type d'exécution n'a pas été changé.


In [ ]:
import torch

print('GPU disponible :', torch.cuda.is_available())
print('Modèle         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')


## 2. Installer les outils

Une minute environ. Les avertissements en rouge sont normaux.


In [ ]:
!pip install -q ultralytics roboflow


## 3. Télécharger le jeu de données

**Coller la clé API Roboflow entre les guillemets ci-dessous.** C'est la seule chose à modifier dans ce notebook.

Cette clé est personnelle. Ne jamais la publier sur GitHub.


In [ ]:
CLE_API_ROBOFLOW = "colle_ta_cle_ici"

from roboflow import Roboflow

rf = Roboflow(api_key=CLE_API_ROBOFLOW)
project = rf.workspace('myworkspace-0p4nk').project('drone-bird-detection-3nl79')
version = project.version(3)
dataset = version.download('yolov11')

print('Téléchargé dans :', dataset.location)


## 4. Contrôler ce qui a été téléchargé

Attendu : classes `['Bird', 'Drone']`, et environ 5418 / 1547 / 772 images.

Si les chiffres ne correspondent pas, ce n'est pas la bonne version du jeu de données.


In [ ]:
import yaml, glob

with open(f'{dataset.location}/data.yaml') as f:
    cfg = yaml.safe_load(f)

print('Classes          :', cfg['names'])
print('Nombre de classes:', cfg['nc'])
print()
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{dataset.location}/{split}/images/*'))
    print(f'{split:6} : {n} images')


## 5. Regarder quelques images

Trente secondes, mais c'est ce qui dit si les annotations valent quelque chose.
Les cadres verts sont les annotations fournies avec le jeu.


In [ ]:
import os, random, glob, cv2
import matplotlib.pyplot as plt

noms = cfg['names']
images = random.sample(glob.glob(f'{dataset.location}/train/images/*'), 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, chemin in zip(axes.ravel(), images):
    img = cv2.cvtColor(cv2.imread(chemin), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    etiquettes = chemin.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    titre = []
    if os.path.exists(etiquettes):
        for ligne in open(etiquettes):
            c, xc, yc, bw, bh = ligne.split()
            xc, yc, bw, bh = float(xc)*w, float(yc)*h, float(bw)*w, float(bh)*h
            p1 = (int(xc - bw/2), int(yc - bh/2))
            p2 = (int(xc + bw/2), int(yc + bh/2))
            cv2.rectangle(img, p1, p2, (0, 255, 0), 3)
            titre.append(noms[int(c)])
    ax.imshow(img); ax.axis('off'); ax.set_title(', '.join(titre) or 'aucune annotation')
plt.tight_layout(); plt.show()


## 6. Entraîner

Vingt à quarante minutes. **Ne pas fermer l'onglet**, Colab coupe la session sinon.

`patience=15` arrête l'entraînement si rien ne s'améliore pendant 15 passes.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
resultats = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    patience=15,
    project='drone_bird',
    name='baseline_n',
)


## 7. Mesurer sur le jeu de test

L'entraînement affiche des scores sur la validation. Le chiffre honnête, celui du README,
se mesure sur le jeu de test, que le modèle n'a jamais vu.

**C'est cette sortie qu'il faut garder.** Référence publiée sur ce jeu : mAP50 = 0.979.


In [ ]:
metrics = model.val(split='test')

print()
print('=== RESULTAT SUR LE JEU DE TEST ===')
print('mAP50    :', round(float(metrics.box.map50), 4))
print('mAP50-95 :', round(float(metrics.box.map), 4))
print()
for i, nom in enumerate(cfg['names']):
    print(f'  {nom:6} mAP50 = {round(float(metrics.box.ap50[i]), 4)}')


## 8. Récupérer le travail avant que la session meure

Colab efface tout à la fermeture. À faire tout de suite.

Tout part dans un seul zip, qui arrive dans le dossier Téléchargements du PC.
Le décompresser ensuite dans `resultats/` et `modeles/` du projet.


In [ ]:
import shutil, os
from google.colab import files

# Ultralytics range ses sorties sous runs/detect/, pas directement sous project/.
base = '/content/runs/detect'
dossier = f'{base}/drone_bird/baseline_n'

assert os.path.exists(dossier), f'introuvable : {dossier}'
print('Contenu :', os.listdir(dossier))

model.export(format='onnx')

# Un seul zip, tout dedans : poids, courbes, matrices de confusion,
# et les sorties de la validation sur le jeu de test.
shutil.make_archive('/content/resultats_baseline_n', 'zip', base)
files.download('/content/resultats_baseline_n.zip')
